<a href="https://colab.research.google.com/github/alejocast2511/LENGUAJE-DE-PROGRAMACION-DCS/blob/main/ESCENARIOS_DE_INVERSION_DESCUBRIENDO_TU_PERFIL_INVERSOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
"""
Proyecto: Análisis de portafolios por Simulación Monte Carlo
Formato: Script preparado para ejecutarse en Google Colab (o local) y desplegar resultados con Streamlit.
Instrucciones rápidas:
1) En Colab: ejecutar las celdas en orden. Para ver la app Streamlit en Colab se usa pyngrok.
2) Librerías: yfinance, pandas, numpy, matplotlib, streamlit, scipy, pyngrok.

Estructura del archivo:
- Instalación e importaciones
- Definición de carteras
- Funciones utilitarias
- Cuestionario + lógica Monte Carlo
- Ejecución de Streamlit vía pyngrok
"""

# ==== (1) Instalación de dependencias (ejecutar en Colab) ====
# En Colab, descomenta y ejecuta:
# !pip install yfinance streamlit pyngrok scipy pandas numpy matplotlib

# ==== (2) Importar librerías ====
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import streamlit as st
from pyngrok import ngrok
import subprocess
import threading
import io

# ==== (3) Definición de carteras actualizadas ====
CARTERAS = {
    'Conservadora': {
        'tickers': ['KO', 'PG', 'JNJ', 'TLT', 'GLD'],  # Consumo básico, salud, bonos, oro
        'pesos': [0.25, 0.25, 0.20, 0.20, 0.10]
    },
    'Balanceada': {
        'tickers': ['AAPL', 'MSFT', 'VTI', 'VNQ', 'DBC'],  # Tecnología, índice global, inmuebles, commodities
        'pesos': [0.20, 0.20, 0.30, 0.15, 0.15]
    },
    'Arriesgada': {
        'tickers': ['TSLA', 'NVDA', 'META', 'BTC-USD', 'SPY'],  # Alto crecimiento y criptomoneda
        'pesos': [0.25, 0.25, 0.20, 0.15, 0.15]
    }
}

# ==== (4) Funciones utilitarias ====
def descargar_datos(tickers, periodo='5y'):
    data = yf.download(tickers, period=periodo, interval='1d', progress=False)['Adj Close']
    if isinstance(data, pd.Series):
        data = data.to_frame()
    data = data.dropna(how='all')
    return data

def calcular_retornos_log(precios):
    return np.log(precios / precios.shift(1)).dropna()

def simulacion_montecarlo(retornos, pesos, dias_horizonte=252, n_simul=5000, pt_inicial=100000):
    mu = retornos.mean().values * 252
    sigma = retornos.cov().values * 252
    chol = np.linalg.cholesky(sigma)

    n_assets = len(mu)
    resultados_final = np.zeros(n_simul)
    trayectorias = np.zeros((n_simul, dias_horizonte))

    for i in range(n_simul):
        z = np.random.normal(size=(dias_horizonte, n_assets))
        shocks = z.dot(chol.T)
        dt = 1/252
        series = np.exp((mu/252 - 0.5 * np.diag(sigma)/252) * 1 + shocks * np.sqrt(dt))
        precios_relativos = np.cumprod(series, axis=0)
        valores = (precios_relativos * pesos).sum(axis=1) * pt_inicial
        resultados_final[i] = valores[-1]
        trayectorias[i, :] = valores

    return {
        'final_values': resultados_final,
        'trayectorias': trayectorias,
        'mean_final': np.mean(resultados_final),
        'median_final': np.median(resultados_final),
        'std_final': np.std(resultados_final),
        'percentiles': np.percentile(resultados_final, [5,25,50,75,95])
    }

# ==== (5) Cuestionario y perfil ====
def determinar_perfil(respuestas):
    horizonte = respuestas.get('horizonte', 5)
    aversion = respuestas.get('aversion_perdida', 3)
    objetivo = respuestas.get('objetivo', 'crecimiento')
    tolerancia = respuestas.get('tolerancia_vol', 'media')

    score = 0
    if horizonte >= 10:
        score += 2
    elif horizonte >=5:
        score += 1
    score += (3 - aversion)
    if objetivo == 'preservacion':
        score -= 2
    if tolerancia == 'baja':
        score -= 1
    elif tolerancia == 'alta':
        score += 1

    if score <= -1:
        return 'Conservadora'
    elif score <= 2:
        return 'Balanceada'
    else:
        return 'Arriesgada'

# ==== (6) App Streamlit ====
def run_app():
    st.title('Simulación Monte Carlo de Portafolios de Inversión')
    st.write('Identifica tu perfil y simula escenarios de inversión basados en carteras reales.')

    st.sidebar.header('Cuestionario del Inversor')
    horizonte = st.sidebar.selectbox('Horizonte (años)', [1,3,5,10,15,20], index=2)
    aversion = st.sidebar.slider('Aversión a pérdidas (1-5)', 1, 5, 3)
    objetivo = st.sidebar.selectbox('Objetivo', ['crecimiento', 'ingresos', 'preservacion'], index=0)
    tolerancia = st.sidebar.selectbox('Tolerancia a la volatilidad', ['baja','media','alta'], index=1)
    monto = st.sidebar.number_input('Monto inicial (USD)', min_value=1000, value=100000, step=1000)

    respuestas = {
        'horizonte': horizonte,
        'aversion_perdida': aversion,
        'objetivo': objetivo,
        'tolerancia_vol': tolerancia
    }

    perfil = determinar_perfil(respuestas)
    st.sidebar.markdown(f'**Perfil estimado:** {perfil}')

    periodo_hist = st.sidebar.selectbox('Periodo histórico', ['1y','3y','5y','10y'], index=2)
    n_simul = st.sidebar.slider('Número de simulaciones', 1000, 20000, 5000, step=500)
    dias_horizonte = st.sidebar.slider('Horizonte (días)', 30, 2520, 252, step=30)

    if st.button('Ejecutar simulación'):
        cartera = CARTERAS[perfil]
        st.write('Cartera seleccionada:', perfil)
        st.write('Activos:', cartera['tickers'])

        with st.spinner('Descargando datos...'):
            precios = descargar_datos(cartera['tickers'], periodo=periodo_hist)
        st.write('Datos desde:', precios.index.min().date(), 'hasta', precios.index.max().date())

        retornos = calcular_retornos_log(precios)
        pesos = np.array(cartera['pesos']) / np.sum(cartera['pesos'])

        with st.spinner('Simulando escenarios...'):
            resultado = simulacion_montecarlo(retornos, pesos, dias_horizonte, n_simul, monto)

        st.subheader('Resultados estadísticos')
        st.write(f"Valor medio final: ${resultado['mean_final']:,.2f}")
        st.write(f"Mediana final: ${resultado['median_final']:,.2f}")
        p = resultado['percentiles']
        st.write(f"Percentiles (5,25,50,75,95): ${p[0]:,.2f}, ${p[1]:,.2f}, ${p[2]:,.2f}, ${p[3]:,.2f}, ${p[4]:,.2f}")

        fig, ax = plt.subplots()
        ax.hist(resultado['final_values'], bins=50)
        ax.set_title('Distribución del valor final')
        ax.set_xlabel('Valor final (USD)')
        ax.set_ylabel('Frecuencia')
        st.pyplot(fig)

        fig2, ax2 = plt.subplots()
        for i in range(min(50, resultado['trayectorias'].shape[0])):
            ax2.plot(resultado['trayectorias'][i, :], alpha=0.4)
        ax2.set_title('Trayectorias simuladas (50 ejemplos)')
        ax2.set_xlabel('Días')
        ax2.set_ylabel('Valor del portafolio (USD)')
        st.pyplot(fig2)

        df_final = pd.DataFrame(resultado['final_values'], columns=['final_value'])
        csv = df_final.to_csv(index=False).encode('utf-8')
        st.download_button('Descargar resultados (CSV)', data=csv, file_name='resultados_finales.csv', mime='text/csv')

# ==== (7) Ejecución en Colab vía pyngrok ====
def launch_streamlit_colab():
    ngrok.kill()
    public_url = ngrok.connect(8501)
    print(f"\nAccede a la aplicación en: {public_url.public_url}\n")

    def run():
        subprocess.call(['streamlit', 'run', 'PESCENARIOS_DE_INVERSION_DESCUBRIENDO_TU_PERFIL_INVERSOR.ipynb', '--server.port', '8501'])

    thread = threading.Thread(target=run)
    thread.start()

# ==== (8) Punto de entrada ====
if __name__ == '__main__':
    run_app()


2025-10-22 03:02:23.695 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.696 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.697 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.699 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.700 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.701 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.702 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-22 03:02:23.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar